In [7]:
import os
import sys
from os import path
sys.path.insert(0, "/home/thomasb")
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
#import sat_utils as su
import importlib
#from albatros_analysis.src.utils import orbcomm_utils as outils
from albatros_analysis.src.utils import baseband_utils as butils
#import helper_discrepancies as hd
#importlib.reload(su)
importlib.reload(butils)
#importlib.reload(hd)

<module 'albatros_analysis.src.utils.baseband_utils' from '/home/thomasb/albatros_analysis/src/utils/baseband_utils.py'>

In [8]:
batch_start_ts = 1753200150

In [24]:
#paths
path_taus = f'/scratch/thomasb/batch_{batch_start_ts}/fine_timing/timing_solution_15may.h5'
path_taus_testing = f'/scratch/thomasb/batch_{batch_start_ts}_testing/fine_timing/timing_solution_15may.h5'
path_UTC = f'/scratch/thomasb/batch_{batch_start_ts}/timing_discrepancies/times_all.json'

#get the conversions from spectrum number into UTC time
with open(path_UTC, 'r') as f:
    data_UTC = json.load(f)
    UTC_per_spec = data_UTC['fit']["UTC_per_spec"]
    UTC_offset = data_UTC['fit']["UTC_offset"]
print(UTC_per_spec)
print(UTC_offset)

#makes reading file easier
map_blines = {0:'MARS 1-2', 1:'MARS 1-4', 2:'MARS 1-5', 3:'MARS 1-6', 4:'MARS 1-7', 5:'MARS 1-8'}

# my flow is indexed by the name of the data file
# for now here's all the russian file names
fnames_russian = [
    'data_raw_osamp=64_start=1753204614_end=1753204909_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753210629_end=1753210874_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753215755_end=1753216140_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753216629_end=1753216874_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753221813_end=1753222206_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753222730_end=1753223066_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753240890_end=1753241135_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753246576_end=1753246773_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753252612_end=1753252907_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753264668_end=1753264963_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753270944_end=1753271238_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753276664_end=1753277253_chans=1836:1839.npy', 
    'data_raw_osamp=64_start=1753277632_end=1753277829_chans=1836:1839.npy']

1.638402806904218e-05
1753200128.4140258


In [25]:
#some parameters
nvis = 120
nant = 7
nbl = int((nant-1)*nant/2)
acclen = 1024
osamp = 64
int_spec = osamp*acclen
T_SPECTRA = 4096/250e6
verbose = False
print('nbl:', nbl)
print('BB spectra per vis:', int_spec)

nbl: 21
BB spectra per vis: 65536


In [26]:
with h5py.File(path_taus, 'r') as f:
    for name, obj in f.items():

        #specnum stuff
        start_spec = obj['taus'].attrs['starting_specnum']
        #beware: we want the CENTRAL spectrum number, not the STARTING one, we accumulate through entire integration time
        spectra = np.arange(int(start_spec+int_spec/2), int(start_spec + (nvis+1/2)*int_spec), int_spec)

        #taus stuff
        taus_old = obj['taus'][:]
        taus_new = np.zeros((nbl, nvis))
        bl_ctr = 0
        for i in range(nant):
            if i == 0:
                taus_i = np.zeros(nvis)
            else:
                taus_i = taus_old[i-1,:]
            for j in range(i+1, nant):
                taus_j = taus_old[j-1,:]
                #convention is ref-nref, so delays are already in that form. 
                #sanity check: for ref ant we want the delays to stay same
                taus_new[bl_ctr, :] = taus_j - taus_i

                if verbose:
                    print(f'\nBaseline {bl_ctr}')
                    print('old i', taus_i[0])
                    print('old j', taus_j[0])
                    print('new:', taus_new[bl_ctr,0])
                bl_ctr+=1
        break

old = taus_old[:,0]
print(old)

[ -3055.55434405 -21417.16028755   7768.85815512  -3634.49347479
  -4092.60467501   8420.41185544]


In [28]:
with h5py.File(path_taus_testing, 'r') as f:
    for name, obj in f.items():
        #specnum stuff
        start_spec = obj['taus'].attrs['starting_specnum']
        #beware: we want the CENTRAL spectrum number, not the STARTING one, we accumulate through entire integration time
        spectra = np.arange(int(start_spec+int_spec/2), int(start_spec + (nvis+1/2)*int_spec), int_spec)

        #taus stuff
        taus_old = obj['taus'][:]
        taus_new = np.zeros((nbl, nvis))
        bl_ctr = 0
        for i in range(nant):
            if i == 0:
                taus_i = np.zeros(nvis)
            else:
                taus_i = taus_old[i-1,:]
            for j in range(i+1, nant):
                taus_j = taus_old[j-1,:]
                #convention is ref-nref, so delays are already in that form. 
                #sanity check: for ref ant we want the delays to stay same
                taus_new[bl_ctr, :] = taus_j - taus_i

                if verbose:
                    print(f'\nBaseline {bl_ctr}')
                    print('old i', taus_i[0])
                    print('old j', taus_j[0])
                    print('new:', taus_new[bl_ctr,0])
                bl_ctr+=1


new = taus_old[:,0]
print(new)

[ -3055.76281107 -21417.40682874   7768.92699198  -3634.51672738
  -4092.84513605   8420.37566406]


In [29]:
print(np.abs(old-new))

[0.20846702 0.2465412  0.06883686 0.02325259 0.24046104 0.03619138]


In [ ]:
taus = {}
stds, errs = {}, {}
utc = []
for bline in map_blines.values():
    taus[bline] = []
    stds[bline] = []
    errs[bline] = []

#open the file up
with h5py.File(path_taus, 'r') as f:
    #iterate over all the russian filenames
    for name in fnames_russian:
        #get baseband spectrum where data starts
        start_spectrum = f[name]['taus'].attrs['starting_specnum']
        # get whole list of baseband spectra for pulse
        spectra = np.arange(start_spectrum, start_spectrum + nvis*acclen*osamp, acclen*osamp)
        #turn into UTC times and append to list
        utc.append(spectra*UTC_per_spec + UTC_offset)

        #need to extract from the h5 as array
        taus_all_file = f[name]['taus'][:]
        errs_all_file = f[name]['errs'][()]
        for i, bline in map_blines.items():
            #add them to the list, indexed by the baseline
            taus[bline].append(taus_all_file[i,:])
            errs[bline].append(errs_all_file[i])             #the error on the fit  (certainty of value)
            stds[bline].append(np.std(taus_all_file[i,:]))   #std across pulse (variation of tau)

#so now if you want to read a certain pulse on a certain baseline:
pnum = 0
bline = 'MARS 1-2'
time_solution = taus[bline][pnum]
print(time_solution.shape)
utcs = utc[pnum]
print(utcs.shape)

(120,)
(120,)


6
